# 📖 Notebook 1: Count-Min Sketch

Before we can find the top-K most viewed videos, we need a way to **count** how many times each video has been viewed. The obvious answer is a hash map (Python dictionary) — but at YouTube scale with billions of unique videos, that hash map would need **64+ GB of memory**.

Count-Min Sketch (CMS) is a probabilistic data structure that counts items using a **fixed amount of memory** — regardless of how many unique items you throw at it. The trade-off? It might slightly **overcount**, but it will never undercount.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why exact counting breaks down at scale
- How Count-Min Sketch uses hash functions and a 2D array to approximate counts
- How to build a CMS from scratch in Python
- The relationship between sketch size, accuracy, and memory usage
- How CMS compares to exact counting on real data from our database

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/top-k
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `topk_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import redis
import time
import random
import mmh3  # MurmurHash3 — a fast, non-cryptographic hash function

# Database connection settings (matches docker-compose.yml)
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "topk_demo",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)

def get_redis_client():
    return redis.Redis(**REDIS_CONFIG)

# Test connections
try:
    conn = get_db_connection()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")
    print("   Run: docker-compose up -d")

try:
    r = get_redis_client()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")
    print("   Run: docker-compose up -d")

## 🤔 Why Not Just Use a Dictionary?

Let's start with the obvious approach: count video views in a Python dictionary.

This works perfectly when you have a small number of items. But what happens when you have **billions** of unique video IDs?

In [ ]:
import sys

# Exact counting with a dictionary — simple and correct
exact_counts = {}

# Simulate counting views for different numbers of unique videos
print("📊 Memory Usage of Exact Counting (Python dict)")
print("=" * 55)
print(f"{'Unique Videos':>15} {'Memory':>12} {'Notes'}")
print("-" * 55)

for num_videos in [100, 10_000, 1_000_000]:
    counts = {f"vid_{i:012d}": random.randint(1, 1_000_000) for i in range(num_videos)}
    memory_bytes = sys.getsizeof(counts)
    # Each entry also has key + value overhead
    total_bytes = memory_bytes + sum(
        sys.getsizeof(k) + sys.getsizeof(v) for k, v in counts.items()
    )
    if total_bytes < 1024 * 1024:
        mem_str = f"{total_bytes / 1024:.1f} KB"
    else:
        mem_str = f"{total_bytes / (1024 * 1024):.1f} MB"
    print(f"{num_videos:>15,} {mem_str:>12}")

print()
# Extrapolate to YouTube scale
estimated_gb = 3_600_000_000 * 80 / (1024**3)  # ~80 bytes per entry
print(f"🔮 Estimated memory for 3.6B videos: ~{estimated_gb:.0f} GB")
print()
print("💡 That's more RAM than most servers have!")
print("   We need a smarter approach — enter Count-Min Sketch.")

## 🧠 How Count-Min Sketch Works

Count-Min Sketch uses a **2D array** of counters (called a "sketch") plus **multiple hash functions**.

Think of it like this analogy:

> Imagine you have 5 friends, and you ask each of them to keep a tally of video views.  
> But instead of tracking every video separately, each friend has only 1000 tally slots.  
> Each friend assigns videos to slots using their own personal rule (hash function).  
> Different friends might put the same video in different slots.
> 
> When you want to know how many views `vid_042` got, you ask all 5 friends:  
> "What's the count in the slot where you put vid_042?"  
> You take the **minimum** of all their answers — that's your best estimate.

The "min" in Count-**Min** Sketch comes from this step!

```
                    Columns (width w)
              ┌───┬───┬───┬───┬───┬───┐
  Row 0 (h₀) │ 0 │ 3 │ 0 │ 7 │ 1 │ 0 │  ← hash₀("vid_042") points here
              ├───┼───┼───┼───┼───┼───┤
  Row 1 (h₁) │ 2 │ 0 │ 5 │ 0 │ 0 │ 4 │  ← hash₁("vid_042") points here
              ├───┼───┼───┼───┼───┼───┤
  Row 2 (h₂) │ 0 │ 0 │ 0 │ 3 │ 8 │ 0 │  ← hash₂("vid_042") points here
              └───┴───┴───┴───┴───┴───┘

  estimate("vid_042") = min(7, 5, 3) = 3
```

### Key Properties
- **Never undercounts** — a counter can only go up, so the true count ≤ the estimate
- **May overcount** — other items might hash to the same slot (collision), inflating the count
- **Fixed memory** — the sketch size is fixed regardless of how many unique items you count
- **More rows + columns = more accurate** — but uses more memory

In [ ]:
class CountMinSketch:
    """
    Count-Min Sketch — a probabilistic frequency counter.
    
    Parameters:
        width:  number of columns (more = fewer collisions = more accurate)
        depth:  number of rows / hash functions (more = lower chance of ALL rows colliding)
    
    Memory usage: width × depth × 4 bytes (using 32-bit integers)
    """
    
    def __init__(self, width: int, depth: int):
        self.width = width
        self.depth = depth
        # Create a 2D array of zeros: depth rows × width columns
        self.table = [[0] * width for _ in range(depth)]
    
    def _hash(self, item: str, row: int) -> int:
        """
        Hash an item to a column index for a given row.
        We use MurmurHash3 with a different seed per row
        so each row has a different hash function.
        """
        return mmh3.hash(item, seed=row) % self.width
    
    def add(self, item: str, count: int = 1):
        """
        Record 'count' occurrences of 'item'.
        Increments one counter per row.
        """
        for row in range(self.depth):
            col = self._hash(item, row)
            self.table[row][col] += count
    
    def estimate(self, item: str) -> int:
        """
        Estimate the count of 'item'.
        Returns the MINIMUM across all rows — this is the closest
        to the true count because it's least affected by collisions.
        """
        return min(
            self.table[row][self._hash(item, row)]
            for row in range(self.depth)
        )
    
    def memory_bytes(self) -> int:
        """Approximate memory usage in bytes (4 bytes per counter)."""
        return self.width * self.depth * 4

print("✅ CountMinSketch class defined!")
print()
print("Let's test it with a simple example...")

In [ ]:
# Simple demo: count fruit
cms = CountMinSketch(width=10, depth=3)

# Add some items
cms.add("apple", 5)
cms.add("banana", 3)
cms.add("cherry", 7)
cms.add("apple", 2)  # apple now has 7 total

print("🍎 Simple CMS Demo (width=10, depth=3)")
print("=" * 40)
print(f"  apple:  estimated={cms.estimate('apple'):>3}  (true=7)")
print(f"  banana: estimated={cms.estimate('banana'):>3}  (true=3)")
print(f"  cherry: estimated={cms.estimate('cherry'):>3}  (true=7)")
print(f"  grape:  estimated={cms.estimate('grape'):>3}  (true=0)")
print()
print("💡 With only 10 columns and 3 rows, estimates are already close!")
print("   But 'grape' might show > 0 due to hash collisions.")
print()

# Show the actual table
print("📋 The sketch table (each row uses a different hash function):")
for i, row in enumerate(cms.table):
    print(f"  Row {i} (hash_{i}): {row}")

## 📊 Testing CMS on Real Video Data

Now let's use our CMS on the actual view events from our PostgreSQL database.  
We'll compare the CMS estimates against the exact counts to see how accurate it is.

In [ ]:
# Load all view events from the database
conn = get_db_connection()
cursor = conn.cursor()
cursor.execute("SELECT video_id FROM view_events ORDER BY viewed_at")
view_events = [row[0] for row in cursor.fetchall()]
conn.close()

print(f"📦 Loaded {len(view_events):,} view events from PostgreSQL")
print(f"   Unique videos: {len(set(view_events))}")
print()

# Get exact counts for comparison
exact = {}
for vid in view_events:
    exact[vid] = exact.get(vid, 0) + 1

# Show top 10 by exact count
top_exact = sorted(exact.items(), key=lambda x: x[1], reverse=True)[:10]
print("🏆 Top 10 videos by exact count:")
for rank, (vid, count) in enumerate(top_exact, 1):
    bar = "█" * (count // 50)
    print(f"  {rank:>2}. {vid}: {count:>6} views  {bar}")

In [ ]:
# Now count the same events using CMS and compare
# We'll try different sketch sizes to see the accuracy trade-off

print("📊 CMS Accuracy vs Sketch Size")
print("=" * 70)

configs = [
    (50, 3, "Tiny"),
    (100, 5, "Small"),
    (500, 5, "Medium"),
    (1000, 7, "Large"),
]

for width, depth, label in configs:
    cms = CountMinSketch(width=width, depth=depth)
    
    # Feed all view events into the sketch
    for vid in view_events:
        cms.add(vid)
    
    # Compare estimates vs exact counts
    errors = []
    for vid, true_count in exact.items():
        est = cms.estimate(vid)
        error_pct = ((est - true_count) / true_count) * 100
        errors.append(error_pct)
    
    avg_error = sum(errors) / len(errors)
    max_error = max(errors)
    mem = cms.memory_bytes()
    
    print(f"\n  {label} ({width}×{depth} = {width*depth:,} counters, {mem:,} bytes)")
    print(f"    Avg overcount: {avg_error:>6.1f}%")
    print(f"    Max overcount: {max_error:>6.1f}%")
    
    # Show top 5 comparison
    print(f"    Top 5 comparison:")
    for vid, true_count in top_exact[:5]:
        est = cms.estimate(vid)
        diff = est - true_count
        marker = "✅" if diff == 0 else f"⚠️  +{diff}"
        print(f"      {vid}: exact={true_count:>5}, CMS={est:>5}  {marker}")

print()
print("💡 Key insight: CMS NEVER undercounts (estimates ≥ true count).")
print("   Bigger sketch = fewer collisions = more accurate estimates.")
print("   But even a tiny sketch gives useful approximations!")

## ⚖️ Memory Comparison: Exact vs CMS

Let's see how much memory we save by using CMS instead of a dictionary.  
Remember: at YouTube scale, this is the difference between needing one server and hundreds.

In [ ]:
print("💾 Memory Comparison: Exact Dictionary vs Count-Min Sketch")
print("=" * 65)
print()

# Our lab data
num_unique = len(exact)
dict_memory = sum(sys.getsizeof(k) + sys.getsizeof(v) for k, v in exact.items())
dict_memory += sys.getsizeof(exact)

cms_medium = CountMinSketch(500, 5)
cms_memory = cms_medium.memory_bytes()

print(f"  Our lab ({num_unique} unique videos):")
print(f"    Dictionary: {dict_memory:>10,} bytes ({dict_memory/1024:.1f} KB)")
print(f"    CMS 500×5:  {cms_memory:>10,} bytes ({cms_memory/1024:.1f} KB)")
print(f"    Savings:    {(1 - cms_memory/dict_memory)*100:.0f}%")
print()

# YouTube scale extrapolation
print(f"  YouTube scale (3.6 billion unique videos):")
yt_dict_bytes = 3_600_000_000 * 80  # ~80 bytes per entry
yt_cms_bytes = 10_000_000 * 10 * 4  # 10M × 10 sketch
print(f"    Dictionary: {yt_dict_bytes / (1024**3):>10.1f} GB")
print(f"    CMS:        {yt_cms_bytes / (1024**2):>10.1f} MB")
print(f"    Savings:    {(1 - yt_cms_bytes/yt_dict_bytes)*100:.2f}%")
print()
print("🚀 CMS uses ~400 MB instead of ~268 GB — a 670× reduction!")
print("   That's the difference between one server and a whole cluster.")

## 🔬 Visualizing Hash Collisions

The only source of error in CMS is **hash collisions** — when two different items hash to the same slot.  
Let's visualize how items spread across the sketch and where collisions happen.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

# Build a small CMS so we can visualize it
vis_cms = CountMinSketch(width=20, depth=4)
for vid in view_events:
    vis_cms.add(vid)

# Plot the sketch as a heatmap
fig, ax = plt.subplots(figsize=(12, 3))
im = ax.imshow(vis_cms.table, aspect='auto', cmap='YlOrRd')
ax.set_xlabel('Column (hash bucket)')
ax.set_ylabel('Row (hash function)')
ax.set_title('Count-Min Sketch Heatmap (20×4) — Darker = More Counts')
ax.set_yticks(range(4))
ax.set_yticklabels([f'hash_{i}' for i in range(4)])
plt.colorbar(im, label='Count')
plt.tight_layout()
plt.show()

print("💡 Hot spots (dark cells) indicate hash collisions.")
print("   Multiple videos are mapped to the same slot, inflating the count.")
print("   Increasing width spreads items more evenly → fewer collisions.")

In [ ]:
# Visualize the error distribution
cms_test = CountMinSketch(width=200, depth=5)
for vid in view_events:
    cms_test.add(vid)

errors = []
labels = []
for vid, true_count in sorted(exact.items(), key=lambda x: x[1], reverse=True):
    est = cms_test.estimate(vid)
    error_pct = ((est - true_count) / true_count) * 100
    errors.append(error_pct)
    labels.append(vid)

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['green' if e < 5 else 'orange' if e < 20 else 'red' for e in errors]
ax.bar(range(len(errors)), errors, color=colors)
ax.set_xlabel('Videos (sorted by true view count, most popular on left)')
ax.set_ylabel('Overcount Error (%)')
ax.set_title('CMS Error by Video (200×5 sketch)')
ax.axhline(y=0, color='black', linewidth=0.5)

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='green', label='< 5% error'),
    Patch(facecolor='orange', label='5-20% error'),
    Patch(facecolor='red', label='> 20% error'),
]
ax.legend(handles=legend_elements)
plt.tight_layout()
plt.show()

print("💡 Popular videos (left) tend to have lower % error.")
print("   Why? Because the collision noise is small relative to their large true count.")
print("   This is great for top-K: the videos we care about are the most accurate!")

## 🔗 CMS in Redis

Redis has **built-in Count-Min Sketch** support! In a real system, you'd use Redis CMS  
instead of building your own. Let's try it with our video data.

In [ ]:
r = get_redis_client()

# Create a Count-Min Sketch in Redis
# CMS.INITBYPROB key error_rate probability
#   error_rate: how much overcount we tolerate (0.01 = 1%)
#   probability: chance of exceeding error_rate (0.01 = 1%)
try:
    r.execute_command('CMS.INITBYPROB', 'notebook1:video_cms', 0.01, 0.01)
    print("✅ Created Redis CMS 'notebook1:video_cms'")
except redis.exceptions.ResponseError as e:
    if 'exists' in str(e).lower():
        print("ℹ️  CMS already exists, reusing it")
    else:
        print(f"⚠️  Redis CMS not available: {e}")
        print("   This requires the RedisBloom module.")
        print("   Our Python CMS above works the same way!")

In [ ]:
# Feed view events into the Redis CMS
try:
    # Batch the increments for efficiency
    # CMS.INCRBY key item count [item count ...]
    batch = {}
    for vid in view_events:
        batch[vid] = batch.get(vid, 0) + 1
    
    args = []
    for vid, count in batch.items():
        args.extend([vid, count])
    
    r.execute_command('CMS.INCRBY', 'notebook1:video_cms', *args)
    print(f"✅ Added {len(view_events):,} events to Redis CMS")
    
    # Query the Redis CMS
    print("\n🏆 Redis CMS estimates vs exact counts (top 10):")
    print(f"   {'Video':<12} {'Exact':>7} {'Redis CMS':>10} {'Error':>7}")
    print("   " + "-" * 40)
    
    for vid, true_count in top_exact:
        result = r.execute_command('CMS.QUERY', 'notebook1:video_cms', vid)
        est = result[0]
        err = est - true_count
        marker = "✅" if err == 0 else f"+{err}"
        print(f"   {vid:<12} {true_count:>7} {est:>10} {marker:>7}")

except redis.exceptions.ResponseError:
    print("⚠️  Redis CMS commands not available (needs RedisBloom module).")
    print("   No worries — our Python CMS above demonstrates the same concepts!")

## 🧹 Cleanup

In [ ]:
# Clean up Redis keys
r = get_redis_client()
keys = r.keys("notebook1:*")
if keys:
    r.delete(*keys)
    print(f"🧹 Cleaned up {len(keys)} Redis keys")
else:
    print("🧹 Nothing to clean up")

## 📚 Summary

### Key Takeaways

1. **Exact counting doesn't scale** — at billions of unique items, a hash map needs 100+ GB of RAM
2. **Count-Min Sketch trades accuracy for memory** — it uses a fixed-size 2D array regardless of item count
3. **CMS never undercounts** — it may overcount due to hash collisions, but never undercount
4. **Bigger sketch = more accurate** — increasing width and depth reduces collision probability
5. **Popular items are most accurate** — collision noise is small relative to large true counts, which is ideal for top-K

### CMS Cheat Sheet

| Operation | Time Complexity | Description |
|-----------|----------------|-------------|
| `add(item, count)` | O(depth) | Increment `depth` counters |
| `estimate(item)` | O(depth) | Read `depth` counters, return minimum |
| Memory | O(width × depth) | Fixed, independent of number of unique items |

### Next Up

CMS tells us *approximately how many views* a video has. But we still need to find the **top K** videos efficiently.  
In **Notebook 2**, we'll combine CMS with a **min-heap** to maintain a running top-K from a stream of events.